# 临时脚本

In [ ]:
!cd ../cpp && make clean && make && cd ..

In [ ]:
from tvm_book.tvm_ext.libinfo import load_lib

_LIB_EXT, _LIB_EXT_NAME = load_lib(name="libtvm_ext.so", search_path=["../cpp/outputs/libs"])

In [ ]:
import tvm_book.tvm_ext.testing._ffi_api as ffi_api

In [1]:
import asyncio
import threading
import time

lock = threading.Lock()

def task():
    time.sleep(10)

async def async_task(lock):
    async with lock:
        res = await asyncio.to_thread(task)
    return res

async def main():
    lock = asyncio.Lock()
    tasks = [asyncio.create_task(async_task(lock)) for _ in range(200)]
    await asyncio.gather(*tasks)

await main()

In [2]:
import time
from multiprocessing import Process

def count(to: int):
    start = time.perf_counter()
    counter = 0
    while counter < to:
        counter += 1
    end = time.perf_counter()

    print(f"在 {end - start} 秒内将 counter 增加到 {to}")

if __name__ == '__main__':
    start = time.perf_counter()
    task1 = Process(target=count, args=(100000000,))
    task2 = Process(target=count, args=(100000000,))
    # 启动进程
    task1.start()
    task2.start()
    # 该方法会一直阻塞主进程，直到子进程执行完成，并且 join 方法内部也可以接收一个超时时间
    # 如果子进程在规定时间内没有完成，那么主进程不再等待
    task1.join()
    task2.join()
    end = time.perf_counter()
    print(f"在 {end - start} 秒内完成")

在 3.7164791940012947 秒内将 counter 增加到 100000000在 3.736047467973549 秒内将 counter 增加到 100000000

在 4.0024557540309615 秒内完成


In [36]:
from concurrent.futures import ProcessPoolExecutor
import asyncio
from asyncio.events import AbstractEventLoop

def count(to: int) -> int:
    counter = 0
    while counter < to:
        counter += 1
    return counter

async def main():
    with ProcessPoolExecutor() as pool:
        loop = asyncio.get_running_loop()
        numbers = [1, 3, 5, 22, 100000000]
        # tasks = [await asyncio.gather(*[[loop.run_in_executor(pool, count, n) for n in numbers]]) for _ in range(2)]
        tasks = [asyncio.gather(*[loop.run_in_executor(pool, count, n) for n in numbers]) for _ in range(2)]
        results = await asyncio.gather(*tasks)

In [37]:
await main()